# Electricity

Data Sources:
* Digital Atlas of Australia:
    * [Electricity Transmission Lines, Digital Atlas of Australia](https://digital.atlas.gov.au/datasets/70f23e91102a4d6899a776d093fa08ef_2)
    * [Transmission Substations, Digital Atlas of Australia](https://digital.atlas.gov.au/datasets/d5eae2d7c9e54581a5f19d7e95b9883b_0)
    * [Major Power Stations, Digital Atlas of Australia](https://digital.atlas.gov.au/datasets/3d0f2d1b8aec4e1a8870b03ce11d4405_1)
* ABS:
    * [Australian Statistical Geography Standard (ASGS) Edition 3, Australian Bureau of Statistics](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs-edition-3/jul2021-jun2026/access-and-downloads/digital-boundary-files#downloads-for-gda94-digital-boundary-files)

In [0]:
%run ./0_Setup

In [0]:
%run ./1_Load_Bronze_Layer

# Load Silver Tables for Transmission Network

* Remove extraneous columns from source

* Convert to geometry data types

In [0]:
%sql
CREATE OR REPLACE TABLE geo.electricity.silver_transmission_line AS
SELECT
    objectid,
    featuretyp as feature_type,
    descriptio,
    class,
    name,
    operationa,
    state,
    spatialcon,
    revised,
    ga_guid,
    capacitykv,
    st_lengths,
    length_m,
    st_astext(st_geomfromwkt(geometry)) as geom_4326
FROM geo.electricity.bronze_transmission_line;

In [0]:
%sql
CREATE OR REPLACE TABLE geo.electricity.silver_major_power_station AS
SELECT
  objectid,
  featuretyp,
  descriptio,
  class,
  name,
  operationa,
  owner,
  generation,
  primaryfue,
  primarysub,
  generati_1,
  generatorn,
  locality,
  state,
  spatialcon,
  revised,
  comment_,
  ga_guid,
  x_coordina,
  y_coordina,
  st_astext(st_geomfromwkt(geometry)) as geom_4326
FROM geo.electricity.bronze_major_power_station;

In [0]:
%sql
CREATE OR REPLACE TABLE geo.electricity.silver_transmission_substation AS
SELECT
  objectid,
  featuretyp,
  descriptio,
  class,
  name,
  operationa,
  state,
  spatialcon,
  revised,
  ga_guid,
  voltagekv,
  locality,
  comment_,
  x_coordina,
  y_coordina,
  st_astext(st_geomfromwkt(geometry)) as geom_4326
FROM geo.electricity.bronze_transmission_substation;

# Process Silver Tables

* Use VIC state multipolygon to filter the transmission network to VIC only, including transmission substations (points) and transmission lines (linestrings).

* Linestrings are computationally expensive as it requires testing containment (H3 index) and intersection (ST intersection). The latter is computationally expensive due to execution on a single worker node rather than utilising parallelism.
  * When testing containment, H3 indexes are created as a preprocessing step.
  * Databrick's native ST Functions are used for point-polygon boundary intersection edge cases.

In [0]:
%sql
CREATE OR REPLACE TABLE geo.digital_boundary.silver_au_states_and_territories AS
SELECT
  STE_NAME21 as state,
  AUS_NAME21 as country,
  AREASQKM21 as area_sqkm,
  st_astext(st_geomfromwkt(geometry)) as geom_4326
FROM geo.digital_boundary.bronze_au_states_and_territories;

# Visualise silver tables

Spot check on a sample of rows from silver tables to understand geospatial context

In [0]:
# Visualise transmission lines as linestrings
import folium
from shapely import wkt

m = folium.Map(location=[-37.81, 144.96], zoom_start=8) # Use OpenStreetMap

# Add buildings and addresses to the map (1K rows ea)
df_geom = spark.sql(f"SELECT * FROM {CATALOG}.{ELEC_SCHEMA}.silver_transmission_line TABLESAMPLE (2000 ROWS)").toPandas()
for _, row in df_geom.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326'])).add_to(m)

# Display the map
display(m)


In [0]:
# Visualise major power station as points
import folium
from shapely import wkt

m = folium.Map(location=[-37.81, 144.96], zoom_start=8) # Use OpenStreetMap

# Add buildings and addresses to the map (1K rows ea)
df_geom = spark.sql(f"SELECT * FROM {CATALOG}.{ELEC_SCHEMA}.silver_major_power_station TABLESAMPLE (2000 ROWS)").toPandas()
for _, row in df_geom.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326']), tooltip=row['name']).add_to(m)

# Display the map
display(m)


In [0]:
# Visualise transmission substation as points
import folium
from shapely import wkt

m = folium.Map(location=[-37.81, 144.96], zoom_start=8) # Use OpenStreetMap

# Add buildings and addresses to the map (1K rows ea)
df_geom = spark.sql(f"SELECT * FROM {CATALOG}.{ELEC_SCHEMA}.silver_transmission_substation").toPandas()
for _, row in df_geom.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326']), tooltip=row['name']).add_to(m)

# Display the map
display(m)


In [0]:
# Visualising the transmission network based on a sample of data
import folium
from shapely import wkt

m = folium.Map(location=[-37.81, 144.96], zoom_start=8, tiles='CartoDB positron') # Use an alternative to OpenStreetMap

# Add buildings and addresses to the map (1K rows ea)
df_geom_line = spark.sql(f"SELECT * FROM {CATALOG}.{ELEC_SCHEMA}.silver_transmission_line TABLESAMPLE (1000 ROWS)").toPandas()
df_geom_substation = spark.sql(f"SELECT * FROM {CATALOG}.{ELEC_SCHEMA}.silver_transmission_substation TABLESAMPLE (1000 ROWS)").toPandas()
for _, row in df_geom_line.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326']), tooltip=row['name']).add_to(m)
for _, row in df_geom_substation.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326']), tooltip=row['name']).add_to(m)


# Display the map
display(m)


In [0]:
# EXPLORATION ONLY: Spike the outputs from the INLINE and h3_tessellateaswkb functions
# %sql
#     SELECT
#       name as tn_line_name,
#       -- Array of structs from h3_tessellateaswkb function
#       cellid AS cell,
#       core,
#       chip -- chip is WKB of (line ∩ cell)
#       FROM (
#         -- INLINE turns array into multiple rows per path, accessible by columns: cellid, core, chip available
#         SELECT
#           name,
#           INLINE(h3_tessellateaswkb(geom_4326, 10))
#         FROM geo.electricity.silver_transmission_line
#       );

In [0]:
%sql
WITH vic_cover AS (
  SELECT DISTINCT p.line_name, p.chip, r.state
  FROM geo.electricity.silver_vic_line_chips_r10 p
  JOIN geo.electricity.silver_vic_cover_r10 r
    ON p.cell = r.cell
)
SELECT
  c.line_name,
  COUNT(c.chip) as number_segments
FROM vic_cover c
GROUP BY c.line_name
ORDER BY number_segments DESC;

# Gold Tables

* VIC Transmission Substations

* VIC Transmission Lines

In [0]:
%sql
-- VIC Transmission Substations
-- NOTE: We chose H3 over st_intersects() / st_intersection() due to multiparallelism
CREATE OR REPLACE TABLE geo.electricity.gold_vic_transmission_substation AS
WITH
  tn_sub_h3 AS (
    -- From POINT geometry, general array of cells / index points to H3 at resolution 9 using h3_pointash3
    SELECT
      name as substation_name,
      featuretyp as feature_type,
      class as tn_class,
      revised as revised_ts,
      voltagekv,
      locality,
      x_coordina as x_coord,
      y_coordina as y_coord,
      geom_4326,
      h3_pointash3(geom_4326, 10) AS tn_substation_cell
    FROM geo.electricity.silver_transmission_substation
    WHERE state = 'Victoria'
  ),
  state_h3 AS (
    -- From MULTIPOLYGON geometry, generate array of cells / index points to H3 at resolution 9 using h3_polyfillash3
    SELECT
      state,
      explode(h3_polyfillash3(geom_4326, 10)) AS state_cell
    FROM geo.digital_boundary.silver_au_states_and_territories
    WHERE state = 'Victoria'
  )
  SELECT
    s.state,
    substation_name,
    feature_type,
    tn_class,
    revised_ts,
    voltagekv,
    locality,
    x_coord,
    y_coord,
    geom_4326
  FROM
    tn_sub_h3 t
  JOIN
    state_h3 s
  ON
    t.tn_substation_cell = s.state_cell
;

In [0]:
%sql
-- VIC Transmission Lines: Containment only
-- Identify candidate geometries from H3 equality join based on containment (NOTE: Calculations do not include intersection test of linestring against geometry)

CREATE OR REPLACE TABLE geo.electricity.gold_vic_transmission_line AS WITH silver_vic_cover_r10 AS (
  -- H3 cover cells for polygons (Victoria)
  SELECT
    state,
    explode(h3_coverash3(geom_4326, 10)) AS cell
  FROM
    geo.digital_boundary.silver_au_states_and_territories
  WHERE
    state = 'Victoria'
),
silver_vic_line_chips_r10 AS (
  -- Tessellate transmission lines to per-cell chips
  SELECT
    name AS line_name,
    cellid AS cell,
    core,
    chip -- chip is WKB of (line ∩ cell)
  FROM
    (
      SELECT
        name,
        -- Array of structs are returned from h3_tessellateaswkb() function. INLINE() turns array into multiple rows per path, accessible by columns: cellid, core, chip available
        INLINE(h3_tessellateaswkb(geom_4326, 10))
      FROM
        geo.electricity.silver_transmission_line
      WHERE
        state = 'Victoria'
    )
),
vic_cover AS (
  SELECT
    DISTINCT p.line_name,
    p.chip,
    r.state
  FROM
    silver_vic_line_chips_r10 p
    JOIN silver_vic_cover_r10 r ON p.cell = r.cell
)
SELECT
  c.state,
  c.line_name,
  st_astext(
      -- Clip each linestring to the polygon by intersecting chips per cell to the polygon, then assemble/reconstruct the result as a linestring using st_union_agg()
      st_union_agg(
        st_geomfromwkb(c.chip)
      )
  ) AS geom_4326
FROM
  vic_cover c
GROUP BY
  c.state,
  c.line_name
;

In [0]:
%sql
-- VIC Transmission Lines: Containment + Intersection
-- Identify candidate geometries from H3 equality join based on containment + intersection
-- Inputs:
--   geo.electricity.silver_au_states_and_territories(state, geom_4326) -- polygon geometries
--   geo.electricity.silver_transmission_line(name, geom_4326) -- linestring geometries
CREATE OR REPLACE TABLE geo.electricity.gold_vic_transmission_line AS WITH silver_vic_cover_r10 AS (
  -- H3 cover cells for polygons (Victoria)
  SELECT
    state,
    explode(h3_coverash3(geom_4326, 10)) AS cell
  FROM
    geo.digital_boundary.silver_au_states_and_territories
  WHERE
    state = 'Victoria'
),
silver_vic_line_chips_r10 AS (
  -- Tessellate transmission lines to per-cell chips
  SELECT
    name AS line_name,
    cellid AS cell,
    core,
    chip -- chip is WKB of (line ∩ cell)
  FROM
    (
      SELECT
        name,
        -- Array of structs are returned from h3_tessellateaswkb() function. INLINE() turns array into multiple rows per path, accessible by columns: cellid, core, chip available
        INLINE(h3_tessellateaswkb(geom_4326, 10))
      FROM
        geo.electricity.silver_transmission_line
      WHERE
        state = 'Victoria'
    )
),
vic_cover AS (
  SELECT
    DISTINCT p.line_name,
    p.chip,
    r.state
  FROM
    silver_vic_line_chips_r10 p
    JOIN silver_vic_cover_r10 r ON p.cell = r.cell
),
vic_cover_count AS (
  SELECT
    c.state,
    c.line_name,
    COUNT(c.chip) as number_segments
  FROM
    vic_cover c
  GROUP BY
    c.state,
    c.line_name
  -- TODO: Added this to reduce computation scope since ST functions are very computationally expensive
  HAVING
    number_segments <= 10
)
SELECT
  c_count.state,
  c_count.line_name,
  -- Store reconstructed linestring as WKT
  st_astext(
    -- Clip each linestring to the polygon by intersecting chips per cell to the polygon, then assemble/reconstruct the result as a linestring using st_union_agg()
    st_union_agg(
      st_intersection(
        st_geomfromwkb(c.chip),
        st_geomfromtext(s.geom_4326)
      )
    )
  ) AS geom_4326
FROM
  vic_cover c
  INNER JOIN vic_cover_count c_count ON c.line_name = c_count.line_name
  LEFT JOIN geo.digital_boundary.silver_au_states_and_territories s ON s.state = c_count.state -- Clip each linestring to the polygon by intersecting chips per cell to the polygon
WHERE
  st_intersects(
    st_geomfromwkb(c.chip), -- Transmission line fragment in this cell
    st_geomfromtext(s.geom_4326) -- state geometry
  )
GROUP BY
  c_count.state,
  c_count.line_name;

# Visualise Gold Tables

See filtered version of geospatial data.

In [0]:
# Visualise transmission substations and lines in VIC
import folium
from shapely import wkt

m = folium.Map(location=[-37.81, 144.96], zoom_start=5, tiles='CartoDB positron') # Use an alternative to OpenStreetMap

# Add polygon of VIC state
state_tn = spark.sql(f"SELECT * FROM {CATALOG}.{BOUND_SCHEMA}.silver_au_states_and_territories WHERE state = 'Victoria'").toPandas()
for _, row in state_tn.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326']), tooltip=row['state']).add_to(m)

# Add transmission substation
df_tn = spark.sql(f"SELECT * FROM {CATALOG}.{ELEC_SCHEMA}.gold_vic_transmission_substation").toPandas()
for _, row in df_tn.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326']), tooltip=row['substation_name']).add_to(m)

# Add transmission line
df_tn = spark.sql(f"SELECT * FROM {CATALOG}.{ELEC_SCHEMA}.gold_vic_transmission_line TABLESAMPLE (200 ROWS)").toPandas() # Only a sample is provided here.
for _, row in df_tn.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326']), tooltip=row['line_name']).add_to(m)

display(m)

# Enhancements

* Add source/sink columns by resolving transmission line name with regex, in order to join transmission lines with transmission substation data. See sections around use of [ST_DWithin() and Recursive CTE](https://www.databricks.com/blog/introducing-spatial-sql-databricks-80-functions-high-performance-geospatial-analytics) in the Databricks Blog.

* If you want to filter all geometries to VIC state, understand how to narrow down scope of processing of ST functions when including edge case of linestring exactly intersecting with polygon boundary

* If you are looking for more [granular boundaries](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs-edition-3/jul2021-jun2026#asgs-diagram) then perform additional filters on the dataset by downloading [digital boundary files from the ABS website](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs-edition-3/jul2021-jun2026/access-and-downloads/digital-boundary-files#downloads-for-gda94-digital-boundary-files).